In [ ]:
# Empty for "ruff" extension

# Feature model time benchmark

This notebook focuses on the time required to compute fingerprints by different feature_models.

Todo:

- Test feature model time to fingerprint with different dataset sizes: 100 rows, qqp, something larger?
- Estimate how time grows with data, comparing them with a line graph
- Maybe measure memory used?


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

from src.fingerprint import ShingleParams, create_fingerprints
from src.spark_utils import get_spark

%load_ext autoreload
%autoreload 2

In [ ]:
from src.datasets import read_llm_paraphrases

spark = get_spark()

MODELS = ["shingles (words)", "shingles (chars)", "bag_of_words", "tf-idf"]
# MODELS = ["embeddings", "shingles (words)", "shingles (chars)", "bag_of_words", "tf-idf"]
SHINGLE_WORDS_CONFIG = ShingleParams(shingle_size=3, over_words=True)
SHINGLE_CHARS_CONFIG = ShingleParams(shingle_size=3, over_words=False)
FINGERPRINT_SIZE = 64
SAMPLE_SIZES = [5000, 10000, 50000, 100000, 200000]
# SAMPLE_SIZES = [10, 50, 100, 500]


COL_IN = "data1"
COL_OUT = "fingerprint1"
df = read_llm_paraphrases()

"""
COL_IN = "data"
COL_OUT = "fingerprint"
df = read_c5()
"""

In [ ]:
from src.config import SPARK_SHUFFLE_PARTITIONS


def benchmark(models: list[str], samples: list[int]) -> list[dict]:
    times = []

    for sample_size in samples:
        if sample_size != -1:
            # For some reason, calling limit() on a PySpark DataFrame
            # causes the collapse of the DataFrame partitioning to just one executor.
            # Therefore, it's necessary to repartition the DataFrame to all workers.
            # There's no mention of this in the official docs, afaik.
            df_sample = df.limit(sample_size).repartition(
                numPartitions=int(SPARK_SHUFFLE_PARTITIONS)
            )
        else:
            df_sample = df

        df_sample.cache()
        df_size = df_sample.count()

        for model in models:
            if model == "shingles (words)":
                model_arg = "shingles"
                config = SHINGLE_WORDS_CONFIG
            elif model == "shingles (chars)":
                model_arg = "shingles"
                config = SHINGLE_CHARS_CONFIG
            else:
                model_arg = model
                config = None

            start_time = time.perf_counter()

            df_temp = create_fingerprints(
                df=df_sample,
                model=model_arg,
                input_col=COL_IN,
                output_col=COL_OUT,
                fingerprint_size=FINGERPRINT_SIZE,
                other=config,
            )

            df_temp.write.format("noop").mode("overwrite").save()

            finish_time = time.perf_counter()

            times.append(
                {
                    "model": model,
                    "time": (finish_time - start_time) * 1000,
                    "fingerprint_size": FINGERPRINT_SIZE,
                    "sample_size": df_size,
                }
            )

        df_sample.unpersist()

    return times

In [ ]:
df.persist()
times = benchmark(MODELS, SAMPLE_SIZES)
df.unpersist()

In [ ]:
plot_df = pd.DataFrame(times)
plot_df["time"] = plot_df["time"].round(2)
plot_pivot = plot_df.pivot_table(index="sample_size", columns="model", values="time")
plt.figure(figsize=(10, 6))
plot = sns.lineplot(data=plot_pivot, palette="deep")
# plt.title("Time performance of feature models (without tf-idf)")
plt.title("Time performance of feature models")
plt.xlabel("Size of sample")
plt.ylabel("Time (ms)")
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
spark.stop()